In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

print("✅ Packages loaded successfully")


In [ ]:
import requests
print("✅ Requests package loaded successfully")


In [ ]:
import os, json, requests, glob
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# Read OPENAI_API_KEY from .env
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
assert API_KEY, "OPENAI_API_KEY is not set. Please check your .env file."

# Working directory (where HTML files are stored)
BASE_DIR = Path.cwd() / "raw_html_data_1"  # Modify path if needed
print("Base dir:", BASE_DIR)


In [4]:
ALLOWED_STATUS = {"planned","in-progress","completed","ongoing","unknown"}
ALLOWED_SCALE  = {"site","neighborhood","city","watershed","regional","unknown"}
ALLOWED_IMPACT_TYPE = {"environmental","social","economic","unknown"}

def to_array(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    # Convert comma/semicolon separated string to array
    s = str(x).strip()
    if not s:
        return []
    parts = [p.strip() for p in re.split(r"[;,]\s*", s)]
    return [p for p in parts if p]

def norm_status(s: str | None) -> str:
    if not s: return "unknown"
    s = s.strip().lower()
    if s in ALLOWED_STATUS: return s
    if any(k in s for k in ["plan","proposal","design phase","preparatory"]): return "planned"
    if any(k in s for k in ["completed","finished","concluded","ended"]):     return "completed"
    if "ongoing" in s:                                                        return "ongoing"
    if any(k in s for k in ["in progress","in-progress","implementation","construction","underway"]):
        return "in-progress"
    return "unknown"

def norm_scale(s: str | None) -> str:
    if not s: return "unknown"
    s = s.strip().lower()
    # Simple mapping
    m = {
        "site": "site",
        "neighbourhood": "neighborhood",
        "neighborhood": "neighborhood",
        "city-wide": "city",
        "city": "city",
        "watershed": "watershed",
        "regional": "regional",
    }
    return m.get(s, s if s in ALLOWED_SCALE else "unknown")

def norm_impacts(x):
    """Force impacts into an array of {description, type} objects"""
    if not x:
        return []
    normed = []
    if isinstance(x, list):
        for item in x:
            if isinstance(item, dict):
                desc = str(item.get("description") or "").strip()
                typ  = str(item.get("type") or "unknown").strip().lower()
                typ  = typ if typ in ALLOWED_IMPACT_TYPE else "unknown"
                if desc:
                    normed.append({"description": desc, "type": typ})
            else:
                # If string, wrap as description
                s = str(item).strip()
                if s:
                    normed.append({"description": s, "type": "unknown"})
    else:
        s = str(x).strip()
        if s:
            normed.append({"description": s, "type": "unknown"})
    return normed


In [5]:
def heuristic_title(html: str) -> str | None:
    soup = BeautifulSoup(html, "html.parser")
    if soup.title and soup.title.string:
        t = soup.title.string.strip()
        if t: return t
    h1 = soup.find("h1")
    if h1:
        t = h1.get_text(strip=True)
        if t: return t
    og = soup.find("meta", property="og:title") or soup.find("meta", attrs={"name":"title"})
    if og and og.get("content"):
        return og["content"].strip()
    return None

def normalise_status(s: str | None) -> str:
    if not s:
        return "unknown"
    s = s.strip().lower()
    allowed = {"planned", "in-progress", "completed", "ongoing", "unknown"}
    if s in allowed:
        return s
    # Simple mapping
    if any(k in s for k in ["plan", "proposal", "design phase", "preparatory"]):
        return "planned"
    if any(k in s for k in ["completed", "finished", "concluded", "ended"]):
        return "completed"
    if "ongoing" in s:
        return "ongoing"
    if any(k in s for k in ["in progress", "in-progress", "implementation", "construction", "underway"]):
        return "in-progress"
    return "unknown"

def heuristic_url(html: str) -> str | None:
    soup = BeautifulSoup(html, "html.parser")
    # canonical
    link = soup.find("link", rel=lambda v: v and "canonical" in v.lower())
    if link and link.get("href"):
        return link["href"].strip()
    # og:url
    og = soup.find("meta", property="og:url")
    if og and og.get("content"):
        return og["content"].strip()
    return None


In [6]:
API_URL = "https://api.openai.com/v1/chat/completions"
MODEL = "gpt-4o-mini"  

CONTROLLED_SOLUTION_TYPES = [
  # Urban typology 
  "Urban Green Spaces",
  "Green Roofs",
  "Rain Gardens / Bioswales",
  "Constructed Wetlands",
  "Urban Forests / Tree-Lined Streets",
  "Permeable Pavements with Vegetation",
  "Blue-Green Infrastructure",
  "Urban Agriculture",
  "Riparian Buffer Zones",
  "Retention Ponds / Detention Basins",
  "Pocket Parks / Microgreenspaces",
  "Living Walls / Vertical Gardens",
  "Urban Waterways Revitalisation",
  "Greenbelts and Ecological Corridors",

  # Natural environments 
  "Forest Restoration",
  "Wetland Restoration",
  "River Restoration",
  "Floodplain Reconnection",
  "Coastal Dune Restoration",
  "Mangrove Restoration",
  "Seagrass/Coral Habitat Restoration",
  "Grassland/Savanna Restoration",
  "Peatland Restoration",
  "Re/afforestation",
  "Riverbank Stabilisation"
]

SYSTEM_PROMPT = """You are an information extraction assistant for a Nature-based Solutions (NBS) database.
Input: raw HTML of a single NBS project page.
Task: extract the following fields.


Rules:
- Do NOT guess. If the source does not say, return "unknown" (for strings) or [] (for arrays).
- Keep answers concise and literal from the source.
- Output ONLY a valid JSON object with exactly the keys listed below.

For "solution_types":
- Choose ONLY from this controlled list (case-sensitive, return exact strings):
{CONTROLLED_SOLUTION_TYPES}
- If none clearly apply, return [].
- Do NOT create new categories or paraphrase.

Fields to extract:

1) title
- Project name as given in the source.
- If missing: "unknown".

2) summary
- 2–4 sentences describing project purpose, actions, and context.
- Prefer executive summary or introduction; otherwise the most relevant section.
- If missing: "unknown".

3) status
- Current stage of the project.
- Allowed values: planned | in-progress | completed | ongoing | unknown
- If unclear: "unknown".

4) location_name
- City/region/named site. If multiple, choose the most specific (city or region).
- If missing: "unknown".

5) country
- Plain country name (e.g., "France").
- If missing: "unknown".

6) scale
- Choose one: site | neighborhood | city | watershed | regional | unknown.

7) solution_types
- Broad categories of NBS used.
- Examples: green roofs, urban wetlands, forest restoration, riverbank stabilization.
- Array of strings; if none: [].

8) challenges_addressed
- Main problems targeted (e.g., flooding, urban heat, air pollution, erosion).
- Array of strings; if none: [].

9) health_linkages_primary
- Direct health outcomes (e.g., heat stress reduction, improved air quality, improved mental health, more active mobility).
- Array of strings; if none: [].

10) impacts
- Documented outcomes (environmental, social, or economic).
- Return as an array of objects: { "description": <short text>, "type": environmental | social | economic | unknown }.
- If none: [].

11) governance
- Who is responsible for implementation/maintenance (e.g., municipal government, community-led, public-private partnership).
- If missing: "unknown".

12) url_source
- Link to the original project page/source document found in the HTML (e.g., canonical, og:url).
- If missing: "unknown".

13) environmental_context
- Broad context (e.g., urban, coastal, wetland, forest, agricultural).
- If missing: "unknown".

Output format:
Return ONLY a valid JSON object with exactly these keys:
["title","summary","status","location_name","country","scale","solution_types","challenges_addressed","health_linkages_primary","impacts","governance","url_source","environmental_context"].
"""

def extract_with_gpt(html: str, pre_title: str | None = None) -> dict:
    # If the HTML is too long, truncate to save tokens (optional)
    # html = html[:100_000]

    sys_prompt = SYSTEM_PROMPT
    if pre_title:
        sys_prompt += f'\n\nIf you are uncertain about "title", consider this candidate extracted locally: "{pre_title}".'

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }
    body = {
        "model": MODEL,
        "response_format": {"type": "json_object"},
        "temperature": 0.0,
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"HTML:\n{html}"},
        ],
    }

    resp = requests.post(API_URL, headers=headers, data=json.dumps(body), timeout=60)
    data = resp.json()

    # Error handling (e.g., insufficient balance or API errors)
    if resp.status_code != 200:
        raise RuntimeError(f"API error {resp.status_code}: {json.dumps(data)[:500]}")

    # Parse model response
    raw = data["choices"][0]["message"]["content"]
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback if JSON format is broken
        raise RuntimeError(f"JSON parse failed: {raw[:500]}")

    title = (obj.get("title") or "unknown").strip()
    summary = (obj.get("summary") or "unknown").strip()
    status = normalise_status(obj.get("status"))

    if (not title or title.lower() == "unknown") and pre_title:
        title = pre_title

    return {"title": title, "summary": summary, "status": status}


In [7]:
import re


In [8]:
API_URL = "https://api.openai.com/v1/chat/completions"
MODEL   = "gpt-4o-mini"  

def extract_with_gpt_extended(html: str, pre_title: str | None = None, pre_url: str | None = None) -> dict:
    sys_prompt = SYSTEM_PROMPT
    # Optional hint injection: provide locally extracted title/URL candidates
    if pre_title:
        sys_prompt += f'\n\nHint: A locally extracted candidate for "title" is "{pre_title}".'
    if pre_url:
        sys_prompt   += f'\nHint: A locally extracted candidate for "url_source" is "{pre_url}".'

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }
    body = {
        "model": MODEL,
        "response_format": {"type": "json_object"},
        "temperature": 0.0,
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user",   "content": f"HTML:\n{html}"},
        ],
    }

    resp = requests.post(API_URL, headers=headers, data=json.dumps(body), timeout=60)
    data = resp.json()
    if resp.status_code != 200:
        raise RuntimeError(f"API error {resp.status_code}: {json.dumps(data)[:400]}")

    raw = data["choices"][0]["message"]["content"]
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        raise RuntimeError(f"JSON parse failed: {raw[:500]}")

    # Field enforcement / normalisation
    title   = (obj.get("title") or "unknown").strip()
    summary = (obj.get("summary") or "unknown").strip()
    status  = norm_status(obj.get("status"))

    location_name = (obj.get("location_name") or "unknown").strip()
    country       = (obj.get("country") or "unknown").strip()
    scale         = norm_scale(obj.get("scale"))

    solution_types         = to_array(obj.get("solution_types"))
    challenges_addressed   = to_array(obj.get("challenges_addressed"))
    health_linkages_primary= to_array(obj.get("health_linkages_primary"))
    impacts                = norm_impacts(obj.get("impacts"))

    governance   = (obj.get("governance") or "unknown").strip()
    url_source   = (obj.get("url_source") or "unknown").strip()
    env_context  = (obj.get("environmental_context") or "unknown").strip()

    # Local fallback corrections
    if (not title or title.lower()=="unknown") and pre_title:
        title = pre_title
    if (not url_source or url_source.lower()=="unknown") and pre_url:
        url_source = pre_url

    return {
        "title": title,
        "summary": summary,
        "status": status,
        "location_name": location_name,
        "country": country,
        "scale": scale,
        "solution_types": solution_types,
        "challenges_addressed": challenges_addressed,
        "health_linkages_primary": health_linkages_primary,
        "impacts": impacts,
        "governance": governance,
        "url_source": url_source,
        "environmental_context": env_context
    }


In [ ]:

import glob

files = sorted(glob.glob("/Users/hyunjicho/Library/CloudStorage/GoogleDrive-bluewings75@gmail.com/My Drive/02_Project/LILY Action/NBS/Unacity/raw_html_data_1/*.html"))
print(f"Found {len(files)} HTML files.")
sample_files = files[:20]
sample_files

In [ ]:
import json
from pathlib import Path

EXPECTED_KEYS = [
    "title","summary","status","location_name","country","scale",
    "solution_types","challenges_addressed","health_linkages_primary",
    "impacts","governance","url_source","environmental_context"
]

rows = []  # use a single variable name for clarity
for fp in sample_files:
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()
    pre_t = heuristic_title(html)
    pre_u = heuristic_url(html)
    try:
        rec = extract_with_gpt_extended(html, pre_title=pre_t, pre_url=pre_u)
    except Exception as e:
        rec = {
            "title":"unknown","summary":"unknown","status":"unknown",
            "location_name":"unknown","country":"unknown","scale":"unknown",
            "solution_types":[], "challenges_addressed":[], "health_linkages_primary":[],
            "impacts":[], "governance":"unknown","url_source":"unknown","environmental_context":"unknown",
        }
        # attach error message as metadata only
        rec["_error"] = str(e)

    # enforce schema: check for missing or extra keys
    missing = [k for k in EXPECTED_KEYS if k not in rec]
    extra   = [k for k in rec.keys() if k not in EXPECTED_KEYS and k not in {"_error"}]
    if missing or extra:
        print("⚠️ schema mismatch", Path(fp).name, "missing:", missing, "extra:", extra)

    # add filename as auxiliary metadata
    rec["_source_file"] = Path(fp).name
    rows.append(rec)

# view columns (13 core + 2 meta)
import pandas as pd
df = pd.DataFrame(rows)
df.head()


In [ ]:
import json
from pathlib import Path
import pandas as pd

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# 1) JSON (preserve full structure)
with open(out_dir / "nbs_sample.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

with open(out_dir / "nbs_sample2.ndjson", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Save as line-delimited JSON (NDJSON)
with open(out_dir / "nbs_sample2.ndjson", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 2) CSV (for human inspection)
df = pd.DataFrame(rows)
for col in ["solution_types", "challenges_addressed", "health_linkages_primary", "impacts"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
df.to_csv(out_dir / "nbs_sample2.csv", index=False, encoding="utf-8")

print("✅ Saved JSON, NDJSON, and CSV →", out_dir)